# Experiment 19 — Debridement at every recall, from the treated state

**Why.** The two analyses of debridement at every recall started from a never-diseased site (H = M = A = 0), whereas the paper describes them as starting from established disease:

- Experiment 15F (part F2): reference site, Figure 5C and Figure S8A;
- Experiment 11 (`debridement_by_sign.csv`): tolerance-test (second) site by sign of adaptation, Figure S7B.

This notebook reruns both protocols **from treated sites** (diseased equilibrium at the baseline plaque, then the defined therapy: debridement and four units of perfect plaque control, as in every other analysis of the paper). For comparison, it reruns them from never-diseased sites too.

- **19A** Reference site (baseline 0.20), plaque peaks 0.60 and 0.75, r = 1 and 0.03, instant host and memory times of 167 and 556 days (ε = 0.01, 0.003); 40 visits. *Figures 5C and S8A.*
- **19B** Tolerance-test site (baseline 0.25), α = −0.5, −0.25, 0.25, 0.5, memory 56 days (ε = 0.03) and instant host, at the peaks used in Experiment 11 (0.05 above the higher of the two hosts' periodic thresholds); 60 visits. *Figure S7B.*
- **19C** Convergence: the memory hosts of 19A followed for 300 visits from both starts, to test whether they end on the same cycle.

**Anchors.** The never-diseased runs must reproduce Experiment 15F (values in the numbers ledger and, if the Experiment 15 export is in Drive, the whole F2 table) and Experiment 11 (downloaded from the archived repository).

**Exports** keep the file layout the figure scripts read. `19A_reference_every_recall/F2_debridement_every_recall.csv` has the columns of `15F_repeated_instrumentation/F2_debridement_every_recall.csv`. `19B_sign_every_recall/debridement_by_sign.csv` has the columns of Experiment 11's `debridement_by_sign.csv`. Both hold the treated start only. Files ending in `_by_start` hold both starts.

Runtime on Colab: about 5 minutes. Every model call goes through the archived `periomemory` package.

In [ ]:
# Install the archived model package (skip if already installed)
import importlib, subprocess, sys
try:
    import periomemory  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/CarloGalliUnipr/periodontal-host-memory-model.git"], check=True)

In [ ]:
import io, json, os, platform, shutil, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy
import periomemory as pm

# ----------------------------------------------------------------- where results are saved
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/Biosemiotic_Perio"
NAME = "Experiment_19_Debridement_From_Treated_State_EXPORT"
EXP15_NAME = "Experiment_15_Model_Revision_EXPORT"      # optional, for the full F2 anchor
REPO_RESULTS = "https://github.com/CarloGalliUnipr/periodontal-host-memory-model/raw/main/results/"
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    BASE = DRIVE_DIR
else:
    BASE = os.environ.get("PERIO_BASE", ".")
OUT = os.path.join(BASE, NAME)
if os.path.exists(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT)
print("export folder:", os.path.abspath(OUT))

CHECKS = []
def check(part, what, value, passed=None):
    CHECKS.append(dict(module=part, check=what, value=value, passed=passed))
    print(f"[{part}] {what}: {value}" + ("" if passed is None else f"  -> {'OK' if passed else 'FAILED'}"))

def part_dir(name):
    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True); return d

def save(df, part, fname):
    df.to_csv(os.path.join(part_dir(part), fname), index=False); return df

# ----------------------------------------------------------------- settings
T = 60.0                      # recall interval (units), about 100 days
REF = dict(pm.REFERENCE)      # n = q = 4, K = 0.9, th0 = 0.8, r = 1, a = 1
SECOND = dict(pm.SECOND)      # tolerance-test site: n = q = 4, K = 0.5, th0 = 0.6
UB_REF, UB_SECOND = 0.20, 0.25
NV_REF, NV_SIGN, NV_CONV = 40, 60, 300
PEAKS_REF = [0.60, 0.75]
HOSTS_REF = [("instant", 0.03, True), ("eps=0.01", 0.01, False), ("eps=0.003", 0.003, False)]
ALPHAS_SIGN = [-0.5, -0.25, 0.25, 0.5]
EPS_SIGN = 0.03
STARTS = ["treated", "naive"]
json.dump(dict(T=T, REF=REF, SECOND=SECOND, UB_REF=UB_REF, UB_SECOND=UB_SECOND, NV_REF=NV_REF, NV_SIGN=NV_SIGN,
               NV_CONV=NV_CONV, PEAKS_REF=PEAKS_REF, HOSTS_REF=[h[0] for h in HOSTS_REF], ALPHAS_SIGN=ALPHAS_SIGN,
               EPS_SIGN=EPS_SIGN, K_REGROW=pm.K_REGROW, T_PC=pm.T_PC, DT=pm.DT, UNIT_DAYS=pm.UNIT_DAYS),
          open(os.path.join(OUT, "parameters.json"), "w"), indent=1)
with open(os.path.join(OUT, "software_versions.txt"), "w") as f:
    f.write(f"python {platform.python_version()}\nnumpy {np.__version__}\nscipy {scipy.__version__}\n"
            f"pandas {pd.__version__}\nmatplotlib {matplotlib.__version__}\nperiomemory {pm.__version__}\n")
T0 = time.time()

## Protocol

`every_recall` is the protocol of Experiment 15F (part F2) and of Experiment 11: at the start of every recall interval the community is removed (M → 0); plaque then regrows from zero, u(t) = u_max(1 − e^(−k t)), reaching `peak` at the end of the interval (k = 0.05, T = 60). The inflammation-free fraction of an interval is the fraction of integration steps with H < 0.5.

`start_state` gives the state at the first recall:
- **treated**: diseased (high) equilibrium at the baseline plaque, then the defined therapy (`pm.therapy`: debridement and four units of perfect plaque control), as for every treated site in the paper;
- **naive**: H = M = A = 0, the start used by Experiments 15F and 11.

In [ ]:
def every_recall(m, state, peak, n_visits):
    # Returns: free fraction per visit (n_visits x N), A at the start of each visit (n_visits x N), final state.
    H, M, A, D = [np.array(x, float) for x in state]
    N = H.size
    peak = np.broadcast_to(np.asarray(peak, float), (N,))
    umax = peak / (1 - np.exp(-pm.K_REGROW * T))
    u_fun = lambda t: umax * (1 - np.exp(-pm.K_REGROW * t))
    fr = np.zeros((n_visits, N)); A_start = np.zeros((n_visits, N))
    st = (H, M, A, D)
    for v in range(n_visits):
        st = (st[0], np.zeros(N), st[2], st[3])          # debridement at the recall
        A_start[v] = st[2]
        free = np.zeros(N); cnt = [0]
        def mon(t, H_, M_, A_, D_):
            free[:] += (H_ < 0.5); cnt[0] += 1
        st, _ = m.run(st, u_fun, 0.0, T, monitor=mon)
        fr[v] = free / cnt[0]
    return fr, A_start, st

def start_state(m, p, ub, start, N=1):
    if start == "naive":
        return tuple(np.zeros(N) for _ in range(4))
    hi = pm.eq_state(p, ub, "high", gamma=0.0)
    if hi is None:
        raise ValueError(f"no diseased state at baseline {ub} for {p}")
    st, _, _ = pm.therapy(m, pm.tile(hi, N))
    return st

def trend(fr):
    # Summary of one free-fraction series.
    x = np.arange(2, len(fr) + 1)
    return dict(first_visit=fr[0], visits_2_5=fr[1:5].mean(), mean_last_10=fr[-10:].mean(),
                slope_per_visit=np.polyfit(x, fr[1:], 1)[0], min_visit=fr.min(), max_visit=fr.max())

## 0. Anchors

The never-diseased start must reproduce Experiment 15F (peak 0.75, r = 0.03: first visit 0.8217 and mean of the last ten 0.2283 at ε = 0.01; 0.98 and 0.23 at ε = 0.003; instant host 0.38–0.39) and Experiment 11 (whole curves, downloaded from the archived repository).

In [ ]:
P = "anchors"
p = dict(REF, r=0.03)
for eps, first, last10 in [(0.01, 0.8216666666666667, 0.22833333333333333), (0.003, 0.98, 0.22999999999999998)]:
    m = pm.Model(**p, eps=eps)
    fr, _, _ = every_recall(m, start_state(m, p, UB_REF, "naive"), 0.75, NV_REF)
    check(P, f"15F, naive start, r=0.03, eps={eps}, peak 0.75: first visit (ledger {first:.4f})", round(float(fr[0, 0]), 6),
          abs(fr[0, 0] - first) < 1e-6)
    check(P, f"15F, naive start, r=0.03, eps={eps}, peak 0.75: mean last 10 (ledger {last10:.4f})", round(float(fr[-10:, 0].mean()), 6),
          abs(fr[-10:, 0].mean() - last10) < 1e-6)

# Experiment 11: peaks and curves from the archived export
def get_export(name):
    for z in [os.path.join(BASE, name + ".zip")]:
        if os.path.exists(z):
            return zipfile.ZipFile(z)
    try:
        data = urllib.request.urlopen(REPO_RESULTS + name + ".zip", timeout=120).read()
        return zipfile.ZipFile(io.BytesIO(data))
    except Exception as e:
        print("download failed:", e); return None

z11 = get_export("Experiment_11_Sign_of_Adaptation_EXPORT")
def read_from_zip(z, fname):
    nm = [n for n in z.namelist() if n.endswith(fname)][0]
    return pd.read_csv(z.open(nm))
if z11 is not None:
    THR11 = read_from_zip(z11, "periodic_thresholds_by_sign.csv")
    DEB11 = read_from_zip(z11, "debridement_by_sign.csv")
    PEAKS_SIGN = {}
    for a in ALPHAS_SIGN:      # Experiment 11 rule: 0.05 above the higher of the two hosts' periodic thresholds
        thr = THR11[THR11.a == a].threshold_peak.max()
        PEAKS_SIGN[a] = float(np.round(thr + 0.05, 3)) if np.isfinite(thr) else 0.6
else:
    print("Experiment 11 export not available: using the peaks it reported; the curve anchor is skipped.")
    THR11 = DEB11 = None
    PEAKS_SIGN = {-0.5: 0.905, -0.25: 0.505, 0.25: 0.46, 0.5: 0.445}
check(P, "Experiment 11 peaks by alpha", PEAKS_SIGN, PEAKS_SIGN == {-0.5: 0.905, -0.25: 0.505, 0.25: 0.46, 0.5: 0.445})

## 19A. Reference site, debridement at every recall (Figures 5C and S8A)

In [ ]:
P = "19A_reference_every_recall"
rows, trows = [], []
for r in [1.0, 0.03]:
    p = dict(REF, r=r)
    for host, eps, inst in HOSTS_REF:
        m = pm.Model(**p, eps=eps, instant=inst)
        for start in STARTS:
            st = start_state(m, p, UB_REF, start, len(PEAKS_REF))
            fr, As, _ = every_recall(m, st, PEAKS_REF, NV_REF)
            for j, pk in enumerate(PEAKS_REF):
                for v in range(NV_REF):
                    rows.append(dict(start=start, r=r, host=host, peak=pk, visit=v + 1, free_fraction=fr[v, j],
                                     A_at_visit_start=As[v, j]))
                trows.append(dict(start=start, r=r, host=host, peak=pk, **trend(fr[:, j])))
F2s = save(pd.DataFrame(rows), P, "F2_by_start.csv")
F2t = save(pd.DataFrame(trows), P, "F2_trend_by_start.csv")
# drop-in replacement for 15F_repeated_instrumentation/F2_debridement_every_recall.csv (treated start)
save(F2s[F2s.start == "treated"][["r", "host", "peak", "visit", "free_fraction"]], P, "F2_debridement_every_recall.csv")
print(F2t.round(3).to_string())

# full anchor against the Experiment 15 export, if it is in Drive
f15 = os.path.join(BASE, EXP15_NAME, "15F_repeated_instrumentation", "F2_debridement_every_recall.csv")
if os.path.exists(f15):
    old = pd.read_csv(f15)
    new = F2s[F2s.start == "naive"].merge(old, on=["r", "host", "peak", "visit"], suffixes=("", "_15F"))
    dev = float((new.free_fraction - new.free_fraction_15F).abs().max())
    check(P, f"naive start reproduces the whole 15F F2 table ({len(new)} rows): max deviation", dev, dev < 1e-9 and len(new) == len(old))
else:
    print("Experiment 15 export not found in Drive; full F2 anchor skipped (ledger anchors above still apply).")
inst = F2s[F2s.host == "instant"].pivot_table(index=["r", "peak", "visit"], columns="start", values="free_fraction")
dev = float((inst["treated"] - inst["naive"]).abs().loc[(slice(None), slice(None), slice(2, None))].max())
check(P, "instant host: treated and naive starts identical from visit 2", round(dev, 6), dev < 1e-3)

## 19B. Tolerance-test site by sign of adaptation (Figure S7B)

In [ ]:
P = "19B_sign_every_recall"
rows, trows = [], []
for a in ALPHAS_SIGN:
    p = dict(SECOND, a=a)
    for host, inst in [(f"eps={EPS_SIGN}", False), ("instant", True)]:
        m = pm.Model(**p, eps=EPS_SIGN, instant=inst)
        for start in STARTS:
            fr, As, _ = every_recall(m, start_state(m, p, UB_SECOND, start), PEAKS_SIGN[a], NV_SIGN)
            for v in range(NV_SIGN):
                rows.append(dict(start=start, a=a, host=host, peak=PEAKS_SIGN[a], visit=v, relapse_free_fraction=fr[v, 0],
                                 A_at_visit_start=As[v, 0]))
            trows.append(dict(start=start, a=a, host=host, peak=PEAKS_SIGN[a], **trend(fr[:, 0])))
S7s = save(pd.DataFrame(rows), P, "debridement_by_sign_by_start.csv")
S7t = save(pd.DataFrame(trows), P, "trend_by_start.csv")
# drop-in replacement for Experiment 11 debridement_by_sign.csv (treated start; visits numbered from 0, as there)
save(S7s[S7s.start == "treated"][["a", "host", "peak", "visit", "relapse_free_fraction"]], P, "debridement_by_sign.csv")
print(S7t.round(3).to_string())
if DEB11 is not None:
    new = S7s[S7s.start == "naive"].merge(DEB11, on=["a", "host", "visit"], suffixes=("", "_11"))
    dev = float((new.relapse_free_fraction - new.relapse_free_fraction_11).abs().max())
    check(P, f"naive start reproduces Experiment 11 debridement_by_sign ({len(new)} rows): max deviation", dev,
          dev < 1e-9 and len(new) == len(DEB11))

## 19C. Convergence: do both starts end on the same cycle?

In [ ]:
P = "19C_convergence"
rows = []
for r in [1.0, 0.03]:
    p = dict(REF, r=r)
    for host, eps, inst in HOSTS_REF[1:]:
        m = pm.Model(**p, eps=eps)
        res = {s: every_recall(m, start_state(m, p, UB_REF, s, len(PEAKS_REF)), PEAKS_REF, NV_CONV) for s in STARTS}
        for j, pk in enumerate(PEAKS_REF):
            row = dict(r=r, host=host, peak=pk)
            for s in STARTS:
                fr, As, _ = res[s]
                row.update({f"{s}_last10_at_40": fr[30:40, j].mean(), f"{s}_last10_at_100": fr[90:100, j].mean(),
                            f"{s}_last10_at_300": fr[-10:, j].mean(), f"{s}_A_at_visit_300": As[-1, j]})
            row["first_visit_within_0.005"] = int(np.argmax(np.abs(res["treated"][0][:, j] - res["naive"][0][:, j]) < 0.005)) + 1
            rows.append(row)
CV = save(pd.DataFrame(rows), P, "convergence.csv")
print(CV.round(4).to_string())
dA = float((CV.treated_A_at_visit_300 - CV.naive_A_at_visit_300).abs().max())
dF = float((CV.treated_last10_at_300 - CV.naive_last10_at_300).abs().max())
check(P, "both starts on the same cycle after 300 visits: max difference in A, in free fraction", (round(dA, 6), round(dF, 6)),
      dA < 1e-3 and dF < 1e-3)

## Numbers for the text, overview figure and export

In [ ]:
def get(df, **k):
    g = df
    for c, v in k.items():
        g = g[np.isclose(g[c], v)] if isinstance(v, float) else g[g[c] == v]
    assert len(g) == 1, k
    return g.iloc[0]
N = []
for start in STARTS:
    for r in [0.03, 1.0]:
        for pk in PEAKS_REF:
            for host, _, _ in HOSTS_REF:
                g = get(F2t, start=start, r=r, peak=pk, host=host)
                N.append(dict(item=f"19A {start}, r={r}, peak {pk}, {host}", first_visit=g.first_visit,
                              mean_last_10=g.mean_last_10, slope_per_visit=g.slope_per_visit, min=g.min_visit, max=g.max_visit))
    for a in ALPHAS_SIGN:
        for host in [f"eps={EPS_SIGN}", "instant"]:
            g = get(S7t, start=start, a=a, host=host)
            N.append(dict(item=f"19B {start}, alpha={a}, {host}, peak {g.peak}", first_visit=g.first_visit,
                          mean_last_10=g.mean_last_10, slope_per_visit=g.slope_per_visit, min=g.min_visit, max=g.max_visit))
NUM = pd.DataFrame(N); NUM.to_csv(os.path.join(OUT, "numbers_for_text.csv"), index=False)
print(NUM.round(3).to_string())

# overview figure (diagnostic; the paper figures are drawn by make_figures_JPR.py and make_figures_JPR_SI.py)
COL = {"instant": "#eb6834", "eps=0.01": "#2a78d6", "eps=0.003": "#104281"}
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, pk, ttl in [(axs[0], 0.75, "Figure 5C: reference, peak 0.75"), (axs[1], 0.60, "Figure S8A: reference, peak 0.60")]:
    for host, _, _ in HOSTS_REF:
        for r, ls in [(1.0, "-"), (0.03, "--")]:
            for start, lw, al in [("treated", 1.4, 1.0), ("naive", 0.8, 0.35)]:
                g = F2s[(F2s.start == start) & (F2s.host == host) & (F2s.r == r) & (F2s.peak == pk)].sort_values("visit")
                ax.plot(g.visit, g.free_fraction, ls, color=COL[host], lw=lw, alpha=al)
    ax.set_ylim(0, 1.05); ax.set_xlabel("recall visit with debridement"); ax.set_ylabel("fraction of interval free of inflammation")
    ax.set_title(ttl + "\n(bold: treated start; faint: never-diseased start)", fontsize=8)
ax = axs[2]
CA = {-0.5: "#6250d6", -0.25: "#a49af0", 0.25: "#2a78d6", 0.5: "#104281"}
for a in ALPHAS_SIGN:
    for host, ls in [(f"eps={EPS_SIGN}", "-"), ("instant", "--")]:
        for start, lw, al in [("treated", 1.4, 1.0), ("naive", 0.8, 0.35)]:
            g = S7s[(S7s.start == start) & (S7s.a == a) & (S7s.host == host)].sort_values("visit")
            ax.plot(g.visit, g.relapse_free_fraction, ls, color=CA[a], lw=lw, alpha=al, label=f"α = {a}" if (start == "treated" and ls == "-") else None)
ax.set_ylim(0, 1.05); ax.set_xlabel("recall visit with debridement"); ax.legend(fontsize=7)
ax.set_title("Figure S7B: tolerance-test site (solid memory 56 d, dashed instant)", fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(part_dir("figures"), "fig19_overview.png"), dpi=150); plt.show()

pd.DataFrame(CHECKS).to_csv(os.path.join(OUT, "all_checks.csv"), index=False)
failed = [c for c in CHECKS if c["passed"] is False]
print("checks failed:", len(failed))
with open(os.path.join(OUT, "README.txt"), "w") as f:
    f.write('''Experiment 19 - Debridement at every recall from the treated state
Protocol of Experiment 15F (part F2) and Experiment 11: community removed at the start of every recall interval (T = 60),
plaque regrowing from 0 to the peak at the end of the interval; inflammation-free fraction = fraction of steps with H < 0.5.
Starts: treated = diseased equilibrium at baseline (0.20 reference, 0.25 tolerance-test), then debridement and 4 units
of perfect plaque control (pm.therapy); naive = H = M = A = 0 (the start of Experiments 15F and 11).
19A_reference_every_recall: F2_by_start.csv (both starts), F2_trend_by_start.csv, and F2_debridement_every_recall.csv
  (treated start, same columns as 15F F2; read by make_figures_JPR.py Figure 5C and make_figures_JPR_SI.py Figure S8A).
19B_sign_every_recall: debridement_by_sign_by_start.csv, trend_by_start.csv, and debridement_by_sign.csv (treated start,
  same columns as Experiment 11; read by make_figures_JPR_SI.py Figure S7B).
19C_convergence: memory hosts of 19A followed for 300 visits from both starts.
numbers_for_text.csv: first visit, mean of the last 10 visits, slope, min and max for every series.
all_checks.csv: anchors (Experiment 15F ledger values, full 15F F2 table if available, Experiment 11 curves) and checks.
''')
print(f"elapsed {time.time() - T0:.0f} s")
zip_path = shutil.make_archive(OUT, "zip", os.path.dirname(os.path.abspath(OUT)), os.path.basename(OUT))
print("wrote", zip_path)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("download did not start; the zip is in Drive:", e)